# 🎬 YouTube Shorts ワンクリック自動生成（一括10本対応）

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `GEMINI_API_KEY` | **初回だけ**入力。2回目からは空欄でOK |
| `PEXELS_API_KEY` | **初回だけ**入力。2回目からは空欄でOK |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約10〜15分 |
| 5本 | 約40〜50分 |
| 10本 | 約80〜100分 |


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: ここだけ変える（毎回）                        ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキー（初回だけ入力。2回目以降は空欄のままでOK） ──
GEMINI_API_KEY = ''   # ← 初回だけ: 'AIza....' を貼り付ける
PEXELS_API_KEY = ''   # ← 初回だけ: 'xxxxx...' を貼り付ける

# ── テーマ（毎回変える） ─────────────────────────────────
THEME = 'ダイエット'   # ← ここにテーマを書く
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── 生成する動画の本数 ───────────────────────────────────
VIDEO_COUNT = 10       # ← 1〜10本（10本で約80〜100分）

# ── 動画の長さ ───────────────────────────────────────────
DURATION = 45          # ← 秒数（30〜60）

# ── 画像スタイル ─────────────────────────────────────────
IMAGE_STYLE = 'realistic'    # 写真そのまま
# IMAGE_STYLE = 'anime'        # アニメ風
# IMAGE_STYLE = 'manga'        # 漫画風（白黒）
# IMAGE_STYLE = 'illustration' # イラスト風

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

print('設定内容を確認します...')
print(f'  テーマ      : {THEME}')
print(f'  生成本数    : {VIDEO_COUNT}本')
print(f'  動画の長さ  : {DURATION}秒')
print(f'  画像スタイル: {IMAGE_STYLE}')

from google.colab import drive
drive.mount('/content/drive')

import os, json
CONFIG_PATH = '/content/drive/MyDrive/YouTube_Production/.config.json'
os.makedirs('/content/drive/MyDrive/YouTube_Production', exist_ok=True)

saved = {}
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        saved = json.load(f)

if GEMINI_API_KEY.strip():
    saved['GEMINI_API_KEY'] = GEMINI_API_KEY.strip()
if PEXELS_API_KEY.strip():
    saved['PEXELS_API_KEY'] = PEXELS_API_KEY.strip()

GEMINI_API_KEY = saved.get('GEMINI_API_KEY', '')
PEXELS_API_KEY = saved.get('PEXELS_API_KEY', '')

if not GEMINI_API_KEY:
    raise ValueError('❌ GEMINI_API_KEY が設定されていません。セル1の GEMINI_API_KEY = に貼り付けてください。')
if not PEXELS_API_KEY:
    raise ValueError('❌ PEXELS_API_KEY が設定されていません。セル1の PEXELS_API_KEY = に貼り付けてください。')

with open(CONFIG_PATH, 'w') as f:
    json.dump(saved, f)

print()
print('✅ 設定完了！「ランタイム → すべてのセルを実行」で動画生成が始まります。')

In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg
print('✅ ツールのインストール完了')

In [ ]:
# 【自動】① YouTubeトレンド分析 → ② 切り口を VIDEO_COUNT 個生成（触らなくてOK）

import requests as _req
import json, re

GEMINI_URL = 'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent'

def call_gemini(prompt, tokens=4096):
    res = _req.post(
        f'{GEMINI_URL}?key={GEMINI_API_KEY}',
        json={'contents': [{'parts': [{'text': prompt}]}],
              'generationConfig': {'temperature': 0.8, 'maxOutputTokens': tokens}},
        timeout=120
    )
    if res.status_code != 200:
        raise RuntimeError(f'Gemini APIエラー ({res.status_code}): {res.text[:300]}')
    return res.json()['candidates'][0]['content']['parts'][0]['text']

print(f'🔍 「{THEME}」のYouTubeトレンドを分析中...')
trend = call_gemini(
    f'あなたはYouTubeショート動画のトレンドアナリストです。'
    f'テーマ「{THEME}」（{DURATION}秒）について、バズりやすいフック・構成・差別化のコツを'
    f'台本制作に直接使える形で簡潔にまとめてください。'
)
print('  ✓ トレンド分析完了')

print(f'\n💡 「{THEME}」の切り口を{VIDEO_COUNT}個考案中...')
angles_raw = call_gemini(
    f'テーマ「{THEME}」のYouTube Shortsで、それぞれ異なる視点・ターゲット・切り口の動画タイトルを{VIDEO_COUNT}個考えてください。'
    f'トレンド分析: {trend[:500]}\n\n'
    f'出力形式（番号なし、1行1タイトル、日本語のみ）:\n'
    f'タイトル1\nタイトル2\n...\nタイトル{VIDEO_COUNT}',
    tokens=1024
)

ANGLES = [line.strip() for line in angles_raw.strip().split('\n') if line.strip()][:VIDEO_COUNT]
while len(ANGLES) < VIDEO_COUNT:
    ANGLES.append(f'{THEME}の攻略法 Vol.{len(ANGLES)+1}')

print(f'\n生成する{VIDEO_COUNT}本の切り口:')
for i, a in enumerate(ANGLES):
    print(f'  {i+1:2d}. {a}')
print(f'\n✅ 切り口の決定完了')

In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）
# ※ 1本あたり約8〜10分、10本で約80〜100分かかります

import cv2, numpy as np, subprocess, tempfile, time
from pathlib import Path
from datetime import datetime
from gtts import gTTS

# ── スタイル変換関数 ──
def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

# ── ユーティリティ ──
def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],capture_output=True,text=True)
    if r.returncode!=0: raise RuntimeError(r.stderr[-500:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def clean(t):
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)','',t)
    t = re.sub(r'シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    return re.sub(r'\s+',' ',t).strip()

W,H,FPS = 1080,1920,30
SCENE_COUNT = 4 if DURATION <= 35 else 5 if DURATION <= 50 else 6
BASE_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
safe_theme = re.sub(r'[\\/:*?"<>|]', '_', THEME)[:20]
BATCH_DIR = Path(f'/content/drive/MyDrive/YouTube_Production/{BASE_TS}_{safe_theme}_batch')
BATCH_DIR.mkdir(parents=True, exist_ok=True)

completed = []
failed = []

def make_one_video(idx, angle):
    vid_num = idx + 1
    print(f'\n{'='*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print('='*50)
    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    safe_angle = re.sub(r'[\\/:*?"<>|]', '_', angle)[:30]
    proj = BATCH_DIR / f'{vid_num:02d}_{safe_angle}'
    img_dir = proj / 'images'
    proj.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(exist_ok=True)

    # ── 台本生成 ──
    print('  📝 台本生成中...')
    scene_count_str = '\n'.join([f'シーン{i+1}（' + ['フック','問題提起','本題①','本題②','まとめ','CTA'][i] + '）:（セリフ）' for i in range(SCENE_COUNT)])
    kw_str = '\n'.join([f'scene{i+1}:（英語1〜3語）' for i in range(SCENE_COUNT)])
    time_str = '\n'.join([f'scene{i+1}:（秒数）' for i in range(SCENE_COUNT)])
    raw = call_gemini(
        f'あなたはYouTubeショート動画の台本専門ライターです。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n【目標尺】{DURATION}秒（{SCENE_COUNT}シーン）\n\n'
        f'以下のフォーマットで出力してください。\n'
        f'[台本]\n{scene_count_str}\n'
        f'[キーワード]\n{kw_str}\n'
        f'[秒数配分]\n{time_str}'
    )

    script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw) or type('x',(),({'group':lambda s,n:raw}))()).group(1)
    kw_block   = re.search(r'\[キーワード\]([\s\S]*?)(?=\[秒数配分\])', raw)
    time_block = re.search(r'\[秒数配分\]([\s\S]*?)$', raw)

    keywords = []
    if kw_block:
        for line in kw_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m: keywords.append(m.group(1).strip())
    while len(keywords) < SCENE_COUNT: keywords.append('motivation')

    timings = []
    if time_block:
        for line in time_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(\d+)', line.strip(), re.I)
            if m: timings.append(int(m.group(1)))
    while len(timings) < SCENE_COUNT: timings.append(DURATION // SCENE_COUNT)

    scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
    scene_texts = [s.strip() for s in scene_lines[1:] if s.strip()]
    while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)

    (proj / '台本.txt').write_text(f'タイトル: {angle}\n\n{script_text}', encoding='utf-8')
    print(f'  ✓ 台本完了')

    # ── 画像取得 ──
    print('  🖼 画像取得中...')
    scenes = []
    for i, (kw, dur) in enumerate(zip(keywords[:SCENE_COUNT], timings[:SCENE_COUNT])):
        time.sleep(0.6)
        fn = f'scene_{i+1:02d}.jpg'
        pth = img_dir / fn
        r = _req.get('https://api.pexels.com/v1/search',
            headers={'Authorization': PEXELS_API_KEY},
            params={'query': kw, 'per_page': 1, 'orientation': 'portrait'}, timeout=15)
        if r.status_code == 200:
            photos = r.json().get('photos', [])
            if photos:
                url = photos[0]['src'].get('portrait') or photos[0]['src'].get('large')
                dl = _req.get(url, timeout=30)
                if dl.status_code == 200:
                    pth.write_bytes(dl.content)
        scenes.append({'scene': i+1, 'image': f'images/{fn}', 'keyword': kw,
                       'duration': dur, 'effect': 'zoom_in' if i%2==0 else 'zoom_out',
                       'text': scene_texts[i] if i < len(scene_texts) else ''})
    print(f'  ✓ 画像完了')

    # ── スタイル変換 ──
    if IMAGE_STYLE != 'realistic' and IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            p = proj / s['image']
            if p.exists():
                img = cv2.imread(str(p))
                if img is not None:
                    cv2.imwrite(str(p), fn_style(img))
        print(f'  ✓ スタイル変換完了（{IMAGE_STYLE}）')

    # ── TTS ──
    print('  🎙 音声生成中...')
    wavs = []
    for i, s in enumerate(scenes):
        t = clean(s.get('text', THEME)) or THEME
        mp3, wav = TMP/f'v{i}.mp3', TMP/f'v{i}.wav'
        try:
            gTTS(text=t, lang='ja').save(str(mp3))
            ff('-i', mp3, '-ar', '44100', '-ac', '1', wav)
            wavs.append(wav)
        except Exception as e:
            print(f'  ⚠ シーン{i+1}音声エラー: {e}')

    VOICE = None
    if wavs:
        lf = TMP/'vl.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
        comb, vaac = TMP/'vc.wav', TMP/'v.aac'
        ff('-f','concat','-safe','0','-i',lf,'-c','copy',comb)
        ff('-i',comb,'-c:a','aac','-ar','44100',vaac)
        VOICE = vaac
    print('  ✓ 音声完了')

    # ── 動画生成 ──
    print('  🎬 動画生成中（一番時間がかかります）...')
    clips = []
    for s in scenes:
        img_path = proj / s['image']
        out = TMP / f"c{s['scene']:02d}.mp4"
        fr = int(s['duration']*FPS); st = 0.15/max(fr,1)
        ze = f"min(1+{st:.6f}*on,1.15)" if s['effect']=='zoom_in' else f"if(eq(on,1),1.15,max(1.0,zoom-{st:.6f}))"
        zp = f"zoompan=z='{ze}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={fr}:s={W}x{H}:fps={FPS}"
        sp = f"scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H}"
        if img_path.exists():
            ff('-loop','1','-i',img_path,'-vf',f'{sp},{zp}','-t',str(s['duration']),'-an','-c:v','libx264','-preset','fast','-crf','22','-pix_fmt','yuv420p',out)
        else:
            ff('-f','lavfi','-i',f'color=black:s={W}x{H}:r={FPS}','-t',str(s['duration']),'-c:v','libx264','-preset','fast',out)
        clips.append(out)

    lf2 = TMP/'cl.txt'
    lf2.write_text('\n'.join(f"file '{p}'" for p in clips))
    mg = TMP/'m.mp4'
    ff('-f','concat','-safe','0','-i',lf2,'-c','copy',mg)

    ah = (f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
          f"[V4+ Styles]\nFormat:Name,Fontname,Fontsize,PrimaryColour,OutlineColour,Bold,Outline,Shadow,Alignment,MarginV\n"
          f"Style:Default,Arial,72,&H00FFFFFF,&H00000000,-1,4,1,2,{int(H*0.18)}\n\n"
          f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n")
    ev = []; t = 0.0
    for s in scenes:
        ev.append(f"Dialogue:0,{at(t)},{at(t+s['duration'])},Default,,0,0,0,,{s['keyword'][:20]}")
        t += s['duration']
    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    od = proj / 'output'
    od.mkdir(exist_ok=True)
    OUT = od / f'shorts_{vid_num:02d}_{datetime.now().strftime("%H%M%S")}.mp4'

    if VOICE and VOICE.exists():
        ff('-i',mg,'-i',VOICE,'-vf',f'ass={se}','-map','0:v','-map','1:a',
           '-c:v','libx264','-preset','medium','-crf','20','-c:a','aac','-b:a','192k',
           '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),OUT)
    else:
        ff('-i',mg,'-vf',f'ass={se}','-c:v','libx264','-preset','medium','-crf','20',
           '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),OUT)

    mb = OUT.stat().st_size / 1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f} MB)')
    return str(OUT)

# ── メインループ ──
start_all = time.time()
for idx, angle in enumerate(ANGLES):
    try:
        out_path = make_one_video(idx, angle)
        completed.append((idx+1, angle, out_path))
    except Exception as e:
        print(f'  ❌ エラー: {e}')
        failed.append((idx+1, angle, str(e)))

elapsed = (time.time() - start_all) / 60
print(f'\n{'='*50}')
print(f'🎉 全動画生成完了！')
print(f'  成功: {len(completed)}本 / 失敗: {len(failed)}本')
print(f'  所要時間: {elapsed:.1f}分')
print(f'  保存先: マイドライブ → YouTube_Production → {BATCH_DIR.name}')
print('='*50)

In [ ]:
# 【自動】完成動画一覧を表示（触らなくてOK）
from IPython.display import Video, display, HTML
import shutil

print(f'✅ 完成した動画 ({len(completed)}本):')
for num, angle, path in completed:
    mb = Path(path).stat().st_size / 1_048_576
    print(f'  {num:2d}. [{mb:.1f}MB] {angle}')

if failed:
    print(f'\n❌ 失敗した動画 ({len(failed)}本):')
    for num, angle, err in failed:
        print(f'  {num:2d}. {angle}')
        print(f'      エラー: {err[:100]}')

if completed:
    print('\n▶ 最初の動画をプレビュー:')
    first_path = completed[0][2]
    shutil.copy(first_path, '/content/preview.mp4')
    print(f'  {completed[0][1]}')
    display(Video('/content/preview.mp4', width=360))